# Geocoding Historical Place Names (Estonian Columns)

This notebook estimates latitude and longitude for rows that have `country`, `region`, and `district` values in Estonian.

## Important notes

- Yes, it is possible, but historical names can be ambiguous or changed over time.
- We use multiple query variants and keep a confidence flag (`query_used`).
- We do not overwrite existing `lat`/`lon`; we only fill missing values.
- We use OpenStreetMap Nominatim via `geopy` with polite rate limiting.
- A cache file is used so repeated runs are faster and avoid repeated API calls.

In [ ]:
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas'])
    import pandas as pd

try:
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'geopy'])
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter

try:
    from tqdm import tqdm
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tqdm'])
    from tqdm import tqdm

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

c:\Users\jayak\OneDrive\문서\github\finno_ugric\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
data_dir = Path('.')
input_file = data_dir / 'finno-ugric-object-places.csv'
output_file = data_dir / 'finno-ugric-object-places-geocoded.csv'
cache_file = data_dir / 'geocode_cache_places.csv'

places = pd.read_csv(input_file)

# Normalize empty strings to missing values for location columns.
for col in ['country', 'region', 'district', 'settlement', 'place_norm', 'lat', 'lon']:
    if col in places.columns:
        places[col] = places[col].replace('', pd.NA)

print(f'Rows: {len(places):,}')
print('Columns:', ', '.join(places.columns))
places.head(3)

Rows: 14,621
Columns: object_id, place_norm, country, region, district, District_controlled, local_admin, settlement, Settlement_controlled, lat, lon, coords_source, place_raw


,object_id,place_norm,country,region,district,District_controlled,local_admin,settlement,Settlement_controlled,lat,lon,coords_source,place_raw
0,518063,Knjazpogostski rajoon; Komi Vabariik; Venemaa,Venemaa,Komi Vabariik,Knjazpogostski rajoon,Knjazpogostski rajoon,NaN,NaN,NaN,NaN,NaN,NaN,"Komi Vabariik, Knjazpogostski rajoon"
1,518064,Knjazpogostski rajoon; Komi Vabariik; Venemaa,Venemaa,Komi Vabariik,Knjazpogostski rajoon,Knjazpogostski rajoon,NaN,NaN,NaN,NaN,NaN,NaN,"Komi Vabariik, Knjazpogostski rajoon"
2,518065,Knjazpogostski rajoon; Komi Vabariik; Venemaa,Venemaa,Komi Vabariik,Knjazpogostski rajoon,Knjazpogostski rajoon,NaN,NaN,NaN,NaN,NaN,NaN,"Komi Vabariik, Knjazpogostski rajoon"


In [3]:
def est_clean(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None

def build_query_candidates(row):
    country = est_clean(row.get('country'))
    region = est_clean(row.get('region'))
    district = est_clean(row.get('district'))
    settlement = est_clean(row.get('settlement'))
    place_norm = est_clean(row.get('place_norm'))

    candidates = []

    # Most specific to least specific.
    if settlement and district and region and country:
        candidates.append(f"{settlement}, {district}, {region}, {country}")
    if district and region and country:
        candidates.append(f"{district}, {region}, {country}")
    if district and country:
        candidates.append(f"{district}, {country}")
    if region and country:
        candidates.append(f"{region}, {country}")
    if place_norm:
        # place_norm often has semicolon-separated historical variants.
        candidates.append(place_norm.replace(';', ','))
    if country:
        candidates.append(country)

    # Keep order, remove duplicates.
    seen = set()
    ordered = []
    for q in candidates:
        if q not in seen:
            ordered.append(q)
            seen.add(q)
    return ordered

In [7]:
# Load geocode cache if available.
if cache_file.exists():
    cache = pd.read_csv(cache_file)
    cache = cache.dropna(subset=['query']).drop_duplicates(subset=['query'], keep='last')
else:
    cache = pd.DataFrame(columns=['query', 'lat', 'lon', 'address'])

cache_map = {
    row['query']: (row.get('lat'), row.get('lon'), row.get('address'))
    for _, row in cache.iterrows()
}

geolocator = Nominatim(user_agent='finno_ugric_geocoder')
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1, swallow_exceptions=True)

def geocode_with_cache(query):
    if query in cache_map:
        lat, lon, address = cache_map[query]
        if pd.notna(lat) and pd.notna(lon):
            return float(lat), float(lon), address

    location = geocode(query, language='et', addressdetails=True)
    if location is None:
        cache_map[query] = (pd.NA, pd.NA, pd.NA)
        return None

    result = (float(location.latitude), float(location.longitude), location.address)
    cache_map[query] = result
    return result

In [ ]:
# Preserve original lat/lon and fill missing values only.
places['lat_filled'] = places['lat']
places['lon_filled'] = places['lon']
places['query_used'] = pd.NA
places['geocoded_address'] = pd.NA

missing_mask = places['lat_filled'].isna() | places['lon_filled'].isna()
to_geocode = places[missing_mask].copy()

print(f"Rows needing geocoding: {len(to_geocode):,}")

for idx, row in tqdm(
    to_geocode.iterrows(),
    total=len(to_geocode),
    desc='Geocoding rows',
    unit='row'
 ):
    queries = build_query_candidates(row)
    found = None
    used_query = None

    for q in queries:
        out = geocode_with_cache(q)
        if out is not None:
            found = out
            used_query = q
            break

    if found is not None:
        lat, lon, address = found
        places.at[idx, 'lat_filled'] = lat
        places.at[idx, 'lon_filled'] = lon
        places.at[idx, 'query_used'] = used_query
        places.at[idx, 'geocoded_address'] = address

print('Geocoding pass complete.')

Rows needing geocoding: 13,812


RateLimiter caught an error, retrying (0/2 tries). Called with (*('Ust-Võmi rajoon, Venemaa',), **{'language': 'et', 'addressdetails': True}).
Traceback (most recent call last):
  File "c:\Users\jayak\OneDrive\문서\github\finno_ugric\.venv\lib\site-packages\geopy\adapters.py", line 298, in get_text
    page = self.urlopen(req, timeout=timeout)
  File "C:\Users\jayak\AppData\Local\Programs\Python\Python310\lib\urllib\request.py", line 519, in open
    response = self._open(req, data)
  File "C:\Users\jayak\AppData\Local\Programs\Python\Python310\lib\urllib\request.py", line 536, in _open
    result = self._call_chain(self.handle_open, protocol, protocol +
  File "C:\Users\jayak\AppData\Local\Programs\Python\Python310\lib\urllib\request.py", line 496, in _call_chain
    result = func(*args)
  File "C:\Users\jayak\AppData\Local\Programs\Python\Python310\lib\urllib\request.py", line 1391, in https_open
    return self.do_open(http.client.HTTPSConnection, req,
  File "C:\Users\jayak\AppData\L

KeyboardInterrupt: 

In [ ]:
# Persist cache and enriched output.
cache_out = pd.DataFrame(
    [
        {'query': q, 'lat': vals[0], 'lon': vals[1], 'address': vals[2]}
        for q, vals in cache_map.items()
    ]
)
cache_out.to_csv(cache_file, index=False, encoding='utf-8')

places.to_csv(output_file, index=False, encoding='utf-8')

print(f'Cache saved: {cache_file.name} ({len(cache_out):,} queries)')
print(f'Enriched file saved: {output_file.name}')

In [ ]:
summary = {
    'total_rows': len(places),
    'original_with_coords': int(places['lat'].notna() & places['lon'].notna()).sum(),
    'filled_with_coords': int((places['lat_filled'].notna() & places['lon_filled'].notna()).sum()),
    'newly_geocoded': int(((places['lat'].isna() | places['lon'].isna()) & (places['lat_filled'].notna() & places['lon_filled'].notna())).sum())
}

pd.DataFrame([summary])

In [ ]:
# Inspect a few newly geocoded rows and the exact query used.
new_rows = places[
    (places['lat'].isna() | places['lon'].isna())
    & places['lat_filled'].notna() & places['lon_filled'].notna()
][['object_id', 'country', 'region', 'district', 'settlement', 'query_used', 'lat_filled', 'lon_filled', 'geocoded_address']]

new_rows.head(20)